## Fase 3: Función de Predicción y Preparación para Despliegue

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import tensorflow as tf
import joblib

# Cargar modelo entrenado
model = tf.keras.models.load_model("model.keras", compile=False)


# Cargar scaler entrenado
scaler = joblib.load("scaler.pkl")

# Cargar dataset preparado
df = pd.read_csv("dataset_preparado.csv", parse_dates=["created_at"])
df = df.sort_values(["product_id", "created_at"])


: 

In [34]:
# Variables usadas en el modelo 
FEATURES = [
    "quantity_on_hand",
    "quantity_reserved",
    "reorder_point",
    "optimal_stock_level",
    "average_daily_usage",
    "stock_status",
    "dia_semana",
    "fin_de_semana",
    "category"
]

TARGET = "quantity_available"
N_STEPS = 7



def build_sequence(product_id, target_date):
    df_p = df[df["product_id"] == product_id].copy()
    df_p = df_p.sort_values("created_at")
    
    # Tomamos los últimos N_STEPS días previos
    df_hist = df_p[df_p["created_at"] < target_date].tail(N_STEPS)

    if len(df_hist) < N_STEPS:
        raise ValueError(f"No hay suficientes datos para {product_id}. Se requieren {N_STEPS} días.")

    cols_scaler = FEATURES + [TARGET]

    scaled = scaler.transform(df_hist[cols_scaler])

    seq = scaled[:, :len(FEATURES)]

    return np.expand_dims(seq, axis=0)   # (1, 7, n_features)


In [46]:
def predict_stock(product_id: str, date: str):
    if isinstance(date, str):
        date = datetime.strptime(date, "%Y-%m-%d")
    
    X = build_sequence(product_id, date)
    
    pred_scaled = model.predict(X)[0][0]
    
    # Desescalar (invertir StandardScaler)
    target_idx = scaler.feature_names_in_.tolist().index(TARGET)
    mean = scaler.mean_[target_idx]
    std = np.sqrt(scaler.var_[target_idx])
    
    pred_real = pred_scaled * std + mean

    return {
        "product_id": product_id,
        "fecha_objetivo": date.strftime("%Y-%m-%d"),
        "prediccion_stock": round(float(pred_real), 2)
    }


In [47]:
prod = df["product_id"].unique()[0]
fecha = (df["created_at"].max() + timedelta(days=1)).strftime("%Y-%m-%d")

print(predict_stock(prod, fecha))


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
{'product_id': 'PROD-001', 'fecha_objetivo': '2025-10-20', 'prediccion_stock': 4.96}


In [ ]:
# Fecha objetivo: mañana después del último registro
fecha_objetivo = (df["created_at"].max() + timedelta(days=0)).strftime("%Y-%m-%d")

# Listar productos únicos
productos = df["product_id"].unique()

resultados = []

for p in productos:
    try:
        print(p +" "+ fecha_objetivo)
        pred = predict_stock(p, fecha_objetivo)
        resultados.append(pred)
    except Exception as e:
        print(f"Error con {p}: {e}")

# Convertir a DataFrame bonito
df_pred = pd.DataFrame(resultados)
df_pred


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step


,product_id,fecha_objetivo,prediccion_stock
0,PROD-001,2025-10-19,4.30
1,PROD-002,2025-10-19,99.53
2,PROD-003,2025-10-19,51.50
3,PROD-004,2025-10-19,10.87
4,PROD-005,2025-10-19,137.32
5,PROD-006,2025-10-19,89.00
6,PROD-007,2025-10-19,63.99
7,PROD-008,2025-10-19,79.24
8,PROD-009,2025-10-19,53.79
9,PROD-010,2025-10-19,91.94
